<left>
<table style='margin-top:0px; margin-left:0px;'>
<tr>
  <td><img src='https://raw.githubusercontent.com/worm-portal/WORM-Figures/master/style/worm.png' alt='WORM' title='WORM' width=50/></td>
  <td><h1 style=font-size:30px>Media Speciator</h1><h2>Notebook</h2></td>
</tr>
</table>
</left>
<p><img src="https://res.cloudinary.com/dcqwdxisc/image/upload/f_auto,q_auto/IMG_6490_nhtouf" alt="aqueous environments" border="0"></p>

<p>This notebook applies aqueous chemical speciation to microbial media recipes using an established thermodynamic modeling package (Boyer et al., 2024). Given a solution composition, along with defined pH and temperature, it calculates how ions are distributed among free species and complexes. This provides insight into what is actually bioavailable in the medium and how solution chemistry shifts under different conditions.</p>

<p>Documentation of the AqEquil package for speciation can be found <a href="https://worm-portal.asu.edu/AqEquil-docs/AqSpeciation.html" rel="nofollow">here</a>.</p>

In [ ]:
import aqequil
ae = aqequil.AqEquil(db="WORM")

In [ ]:
# perform a speciation calculation
speciation = ae.speciate(input_filename="example-speciation-input.csv", #formatted_file
                         exclude=["Name","Year"],
                         report_filename="speciation_output.csv", # create a CSV of results 
                         delete_generated_folders=True)
                         #charge_balance_on='HCO3-')

speciation.save("speciation-for-energy") #save file for energy supply step

This plot shows the % mass contribution of complexes containing a given basis species. This species can be changed depending on what you want to look at.

In [ ]:
speciation.plot_mass_contribution('SO4-2', sort_by="Year", ascending=True)

### Energy Supplies

The next code will allow you to calculate the energy supplies of your medium and/or original sample geochemistry data. The top n number of reactions are then printed in tables and plotted as a bar chart. (explain generally how this is calculated)

You can start here any time after speciating.

In [ ]:
#import tools for looking at top energy supplies
from tools import *

In [ ]:
import aqequil
ae = aqequil.AqEquil()
speciation = aqequil.load("speciation-for-energy.speciation")

In [ ]:
#make all possible redox reactions of the half reactions available by default in the package
speciation.make_redox_reactions()
speciation.redox_formatted_reactions.to_csv("formatted_redox_rxns")

In [ ]:
#calculate the energy supplies
speciation.apply_redox_reactions(y_type='E',
                                 y_units='J')
speciation.save("speciation-with-energy")

In [ ]:
#get a file of just the energy supplies for visual analysis
df = speciation.lookup(speciation.lookup('energy supply'))
df.columns = ['_'.join(col).strip() for col in df.columns.values]
df.to_csv("energy-supplies.csv")

In [ ]:
#print the top energy supplies
print_top_energy_supplies(
    energy_file="energy-supplies.csv",
    reaction_file="formatted_redox_rxns",
    sample_name_col="Sample",
    top_n=5 #however many rxns you want to see
)

In [ ]:
#plot the top energy supplies
plot_top_energy_supplies(
    energy_file="energy-supplies.csv",
    reaction_file="formatted_redox_rxns",
    sample_name_col="Sample",
    top_n=5 #however many top rxns you want to see
)